# 30 — Angle metric validation (metric v3 vs expression)

`EvaluationHarness.concept_angles` introduces gauge-invariant Grassmann
angle/distance metrics between a text's hidden states and a concept's span
(`frames/linalg/angles.py`): bounded, scale-invariant, comparable across
models — unlike the expression metric's linear trace correlation, into
which hidden-state norms enter directly.

This notebook validates the new metric **offline on existing logs** — no new
generation. The 1,170 showcase records of the AWQ Llama (notebook 24) and
the unquantized bf16 Llama (notebook 29) are re-scored with their own
models; baseline continuations are regenerated (greedy — deterministic),
and a gate cross-checks reproduction by recomputing the logged baseline
expression values.

Questions answered:

1. **Agreement** — how strongly do angle-deltas correlate with the
   expression-deltas every prior conclusion rests on?
2. **Disagreements** — records with high expression-delta but flat
   angle-delta are candidates where the linear metric was inflated by
   hidden-state norm growth rather than genuine directional movement.
3. **Cross-model payoff** — AWQ vs bf16 compared directly in angle units
   (the comparison raw deltas could not make).


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.utils.memory import gc_cuda

In [ ]:
MIN_LEMMAS = 11
MAX_TOKENS = 3
BASELINE_TOKENS = 26  # STEPS + 2, as in notebooks 24/29

MODELS = {
    "llama-3.1-8b-awq": {
        "log": "resources/24_showcase.jsonl",
        "method_map": {
            "single": "single",
            "F1.a-mean": "blend",
            "F2.a-wsum": "vote",
            "F3.a-roundrobin": "schedule",
        },
        "kwargs": {
            "id": "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
            "device_map": "cuda:0",
        },
    },
    "llama-3.1-8b-bf16": {
        "log": "resources/29_llama8b_bf16.jsonl",
        "method_map": None,  # already descriptive
        "kwargs": {
            "id": "meta-llama/Llama-3.1-8B-Instruct",
            "torch_dtype": torch.bfloat16,
            "device_map": "auto",
        },
    },
}


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

OUT_LOG = Path("resources/30_angle_rescore.jsonl")
OUT_LOG.unlink(missing_ok=True)

In [ ]:
def rescore_model(model_key: str, spec: dict) -> None:
    fur = FrameUnembeddingRepresentation.from_model_id(**spec["kwargs"])
    harness = EvaluationHarness(
        fur, "/dev/null", min_lemmas_per_synset=MIN_LEMMAS, max_token_count=MAX_TOKENS
    )
    prompts = [chat(u, a) for u, a in QUESTIONS]
    baseline_texts = harness.generate_baseline(prompts, max_new_tokens=BASELINE_TOKENS)
    baseline_cont = {
        p: harness.continuation(t, p) for p, t in zip(prompts, baseline_texts)
    }

    records = [
        json.loads(line) for line in Path(spec["log"]).read_text().strip().split("\n")
    ]
    records = [r for r in records if r["expression"] and r["prompt"] in baseline_cont]
    print(f"[{model_key}] {len(records)} records to re-score")

    # gate: regenerated baselines must reproduce the logged baseline
    # expression values (greedy decoding is deterministic per model)
    checked = 0
    for r in records:
        for name, vals in r["expression"].items():
            concept = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
            redo = harness.concept_expression(baseline_cont[r["prompt"]], concept)
            assert abs(redo - vals["baseline"]) < 1e-2, (name, redo, vals["baseline"])
            checked += 1
            break
        if checked == 5:
            break
    print(f"[{model_key}] baseline reproduction gate passed")

    method_map = spec["method_map"]
    with OUT_LOG.open("a") as handle:
        for i, r in enumerate(records):
            concepts = {
                name: fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
                for name in r["expression"]
            }
            angles = harness.angle_record(
                r["continuation"], baseline_cont[r["prompt"]], concepts
            )
            method = r["config"]["method"]
            handle.write(
                json.dumps(
                    {
                        "model": model_key,
                        "method": method_map[method] if method_map else method,
                        "prompt": r["prompt"],
                        "continuation": r["continuation"],
                        "expression": r["expression"],
                        "angles": angles,
                        "ppl_ratio": r["ppl_ratio"],
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )
            if (i + 1) % 200 == 0:
                print(f"[{model_key}] {i + 1}/{len(records)}")


for model_key, spec in MODELS.items():
    with gc_cuda():
        rescore_model(model_key, spec)

In [ ]:
rows = []
for line in OUT_LOG.read_text().strip().split("\n"):
    r = json.loads(line)
    for name, a in r["angles"].items():
        rows.append(
            {
                "model": r["model"],
                "method": r["method"],
                "concept": name,
                "expr_delta": r["expression"][name]["delta"],
                "d_cos": a["delta"]["token_cos"],
                "d_geo": a["delta"]["geodesic"],
                "d_mcc": a["delta"]["mcc"],
                "steered_cos": a["steered"]["token_cos"],
                "continuation": r["continuation"],
                "ppl_ratio": r["ppl_ratio"],
            }
        )
results = pd.DataFrame(rows)
finite = results[["expr_delta", "d_cos", "d_geo", "d_mcc"]].notna().all(axis=None)
assert finite and len(results) > 2000, (len(results), finite)
print(f"gate: {len(results)} concept-level scores, all finite")

print("\ncorrelation of angle deltas with expression delta:")
for model, group in results.groupby("model"):
    for col in ("d_cos", "d_geo", "d_mcc"):
        pearson = group["expr_delta"].corr(group[col])
        spearman = group["expr_delta"].corr(group[col], method="spearman")
        print(f"  {model:18s} {col}: pearson={pearson:+.3f} spearman={spearman:+.3f}")

summary = (
    results.groupby(["model", "method"])
    .agg(
        expr_delta=("expr_delta", "mean"),
        d_cos=("d_cos", "mean"),
        d_geo=("d_geo", "mean"),
        d_mcc=("d_mcc", "mean"),
        steered_cos=("steered_cos", "mean"),
        n=("d_cos", "size"),
    )
    .reset_index()
)
summary.to_csv("resources/30_angle_summary.csv", index=False)
summary.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

colors = {"llama-3.1-8b-awq": "tab:red", "llama-3.1-8b-bf16": "tab:blue"}
for model, group in results.groupby("model"):
    axes[0].scatter(
        group["expr_delta"],
        group["d_cos"],
        s=4,
        alpha=0.25,
        color=colors[model],
        label=model,
    )
axes[0].axhline(0, color="gray", lw=0.8)
axes[0].axvline(0, color="gray", lw=0.8)
axes[0].set_xlabel("expression delta (linear, model-scaled)")
axes[0].set_ylabel("token-cosine delta (bounded, scale-free)")
axes[0].legend(fontsize=8)

order = ["single", "blend", "vote", "schedule"]
for j, model in enumerate(colors):
    vals = [
        summary[(summary["model"] == model) & (summary["method"] == m)]["d_cos"].iloc[0]
        for m in order
    ]
    axes[1].bar(
        [i + (j - 0.5) * 0.35 for i in range(len(order))],
        vals,
        width=0.34,
        color=colors[model],
        label=model,
    )
axes[1].set_xticks(range(len(order)), order)
axes[1].set_ylabel("mean token-cosine delta (comparable across models)")
axes[1].axhline(0, color="gray", lw=0.8)
axes[1].legend(fontsize=8)

plt.suptitle("Angle metric vs expression metric (notebooks 24 + 29 re-scored)")
plt.tight_layout()
plt.savefig("resources/30_angle_vs_expression.png", dpi=150)
plt.show()

In [ ]:
# disagreements: strong expression gain but flat/negative directional movement
hi_expr = results["expr_delta"] > results["expr_delta"].quantile(0.75)
lo_cos = results["d_cos"] < results["d_cos"].quantile(0.25)
suspects = results[hi_expr & lo_cos].sort_values("expr_delta", ascending=False)
print(
    f"{len(suspects)} norm-inflation suspects "
    f"({len(suspects) / len(results):.0%} of scores)\n"
)
for _, row in suspects.head(5).iterrows():
    print(
        f"[{row['model']} / {row['method']} / {row['concept'].split('.')[0]}] "
        f"expr={row['expr_delta']:+.2f} d_cos={row['d_cos']:+.4f}"
    )
    print("   ", row["continuation"][:140].replace("\n", " "), "\n")